### IMPORT AND SETUP

In [1]:
import os
import sys
import torch
import gc
import pandas as pd
import numpy as np

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# Khai báo đường dẫn root để import các module từ thư mục core
sys.path.append(os.path.abspath("../"))

from core.module02_fm_embedding_extraction.models import BioModelManager
from core.module02_fm_embedding_extraction.extractor import FeatureExtractor
from core.module03_geometry_extraction.extractor import LatentGeometryCalculator, GeometryExtractionProfiler
from core.module04_geo_bio_context_normalization.normalizer import AdvancedFeatureNormalizer
from core.module02_fm_embedding_extraction.profiler import FoundationModelProfiler # Import class Profiler

# Giải phóng bộ nhớ GPU trước khi chạy bộ khung trích xuất mới
torch.cuda.empty_cache()
gc.collect()

print(f"[+] CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Đang sử dụng thiết bị: {torch.cuda.get_device_name(0)}")

c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[+] CUDA khả dụng: True
[+] Đang sử dụng thiết bị: NVIDIA GeForce RTX 5080


### Matrix Configurations

In [2]:
# Set up cho Module 2
# 1. Danh sách các file Parquet cần chạy (Hỗ trợ Train, Val, Test gom trong 1 lần cắm máy)
INPUT_FILES_CONFIG = [
    {
        "input_path": "D:/variant_data/train1_full_seq_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/train1"
    },
    {
        "input_path": "D:/variant_data/train2_full_seq_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/train2"
    },
    {
        "input_path": "D:/variant_data/train3_full_seq_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/train3"
    },
    {
        "input_path": "D:/variant_data/val_full_seq_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/val"
    },
    {
        "input_path": "D:/variant_data/test_full_seq_after_vep_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/test"
    },
    {
        "input_path": "D:/variant_data/clinvarhq_full_seq_after_vep_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/clinvarhq"
    },
    {
        "input_path": "D:/variant_data/uniprot_full_seq_after_vep_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/uniprot"
    },
    {
        "input_path": "D:/variant_data/proteingym_full_seq_after_vep_final.parquet",
        "output_dir": "D:/variant_data/fm_embeddings/proteingym"
    }
]

# 2. Không gian ma trận các mô hình mục tiêu cần trích xuất đặc trưng
# Bắt buộc khai báo chuẩn xác model_id trên HuggingFace và phân loại kiến trúc học (mlm / causal)
MODELS_SPACE = [
    {
        "name": "nt_v1_500m",
        "model_id": "InstaDeepAI/nucleotide-transformer-500m-human-ref",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 64
    },
    {
        "name": "nt_v3_650m",
        "model_id": "InstaDeepAI/NTv3_650M_pre",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 32
    },
    {
        "name": "nt_v2_500m",
        "model_id": "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 32
    },
    {
        "name": "esm1b_650m",
        "model_id": "facebook/esm1b_t33_650M_UR50S",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esm2_650m",
        "model_id": "facebook/esm2_t33_650M_UR50D",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esmc_600m",
        "model_id": "biohub/ESMC-600M-hf",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    }
]

# Strict mode A (0-based): mutation luôn ở vị trí cố định
STRICT_MODE = True
STRICT_FIXED_POS = {"dna": 300, "protein": 50}

# Resume mode: nếu True thì skip các output đã tồn tại
RESUME_MODE = True

# Split chuẩn cho toàn pipeline
TRAIN_SPLITS = ["train1", "train2", "train3"]
VAL_SPLITS = ["val"]
TEST_SPLITS = ["test", "clinvarhq", "uniprot", "proteingym"]
ALL_SPLITS = TRAIN_SPLITS + VAL_SPLITS + TEST_SPLITS

# Set up cho Module 3
BASE_DIR = r"D:/variant_data"
EMB_DIR = f"{BASE_DIR}/fm_embeddings"
INDEX_DIR = f"{BASE_DIR}/faiss_indexes"
GEOM_DIR = f"{BASE_DIR}/geometry"

MODELS = ["nt_v1_500m", "nt_v3_650m", "nt_v2_500m", "esm1b_650m", "esm2_650m", "esmc_600m"]
POOLINGS = ["cls", "center", "mean"]

os.makedirs(INDEX_DIR, exist_ok=True)
for split in ALL_SPLITS:
    os.makedirs(f"{GEOM_DIR}/{split}", exist_ok=True)

# Set up cho Module 4
INPUT_DIR = BASE_DIR
OUTPUT_DIR = f"{BASE_DIR}/processed_parquet"
ARTIFACTS_DIR = f"{BASE_DIR}/normalization_artifacts"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Bio parquet mapping
BIO_TRAIN_PATHS = [
    f"{INPUT_DIR}/train1_full_seq_final.parquet",
    f"{INPUT_DIR}/train2_full_seq_final.parquet",
    f"{INPUT_DIR}/train3_full_seq_final.parquet",
]
BIO_EVAL_PATHS = [
    f"{INPUT_DIR}/val_full_seq_final.parquet",
    f"{INPUT_DIR}/test_full_seq_after_vep_final.parquet",
    f"{INPUT_DIR}/clinvarhq_full_seq_after_vep_final.parquet",
    f"{INPUT_DIR}/uniprot_full_seq_after_vep_final.parquet",
    f"{INPUT_DIR}/proteingym_full_seq_after_vep_final.parquet",
]

# MODULE 2: FOUNDATION MODELS EMBEDDING EXTRACTION

In [3]:
FM_PROFILE_JSON = r"D:/variant_data/profiling/fm_profiling.json"

PROFILE_SPLITS = {"test", "clinvarhq", "uniprot", "proteingym"}

def get_split_name(path: str) -> str:
    name = os.path.basename(path).lower()
    if name.startswith("train1"):
        return "train1"
    if name.startswith("train2"):
        return "train2"
    if name.startswith("train3"):
        return "train3"
    if name.startswith("val"):
        return "val"
    if name.startswith("test"):
        return "test"
    if name.startswith("clinvarhq"):
        return "clinvarhq"
    if name.startswith("uniprot"):
        return "uniprot"
    if name.startswith("proteingym"):
        return "proteingym"
    return "unknown"

def _expected_embedding_paths(output_prefix: str):
    return [f"{output_prefix}_cls.pt", f"{output_prefix}_center.pt", f"{output_prefix}_mean.pt"]

def _all_exists(paths):
    return all(os.path.exists(p) for p in paths)

def _validate_fixed_position(df, seq_col_ref, seq_col_alt, fixed_idx, seq_label, file_path):
    required_cols = ["Variant_ID", seq_col_ref, seq_col_alt]
    miss = [c for c in required_cols if c not in df.columns]
    if miss:
        raise ValueError(f"[{file_path}] Thiếu cột bắt buộc cho {seq_label}: {miss}")

    ref = df[seq_col_ref].astype(str)
    alt = df[seq_col_alt].astype(str)

    short_mask = (ref.str.len() <= fixed_idx) | (alt.str.len() <= fixed_idx)
    len_mismatch = ref.str.len() != alt.str.len()
    same_char = ref.str.get(fixed_idx) == alt.str.get(fixed_idx)

    bad_mask = short_mask | len_mismatch | same_char
    n_bad = int(bad_mask.sum())

    print(f"[Precheck:{seq_label}] {os.path.basename(file_path)} -> tổng {len(df)} | lỗi {n_bad}")
    if n_bad > 0:
        preview = df.loc[bad_mask, ["Variant_ID", seq_col_ref, seq_col_alt]].head(5)
        print("  [Chi tiết 5 dòng đầu lỗi]")
        print(preview.to_string(index=False))
        raise RuntimeError(
            f"Strict precheck thất bại cho {os.path.basename(file_path)} ({seq_label}): {n_bad} mẫu không đạt. Dừng pipeline."
        )

def strict_precheck_all_inputs(input_cfgs, fixed_pos):
    print("=" * 80)
    print("[*] STRICT PRECHECK: quét toàn bộ input trước khi chạy extraction")
    print("    - Quy ước index: 0-based")
    print(f"    - DNA fixed position: {fixed_pos['dna']}")
    print(f"    - Protein fixed position: {fixed_pos['protein']}")
    print("=" * 80)

    for cfg in input_cfgs:
        file_path = cfg["input_path"]
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Thiếu input parquet: {file_path}")

        df = pd.read_parquet(file_path)
        _validate_fixed_position(df, "ref_seq", "alt_seq", fixed_pos["dna"], "dna", file_path)
        _validate_fixed_position(df, "prot_ref_seq", "prot_alt_seq", fixed_pos["protein"], "protein", file_path)

    print("[+] STRICT PRECHECK PASS: toàn bộ dataset đạt điều kiện fixed-position.\n")

if STRICT_MODE:
    strict_precheck_all_inputs(INPUT_FILES_CONFIG, STRICT_FIXED_POS)

for model_cfg in MODELS_SPACE:
    print("=" * 80)
    print(f"[MÔ HÌNH HIỆN TẠI]: KHỞI CHẠY TIẾN TRÌNH TRÍCH XUẤT CHO MÔ HÌNH: {model_cfg['name'].upper()}")
    print("=" * 80)
    
    try:
        model_manager = BioModelManager(
            model_id=model_cfg["model_id"],
            model_type=model_cfg["model_type"],
            device="cuda"
        )
        extractor = FeatureExtractor(model_manager=model_manager)

        profiler = FoundationModelProfiler(model_cfg['name'], torch.device("cuda"))
        profiler.measure_static_memory(model_manager.model)

        strategy = model_manager.probe_token_mapping_strategy(model_cfg["seq_type"])
        print(f"[Token Strategy] {model_cfg['name']}: {strategy}")
        
    except Exception as e:
        print(f"[LỖI CHIẾN LƯỢC] Không thể nạp mô hình {model_cfg['name']}. Lỗi chi tiết: {e}")
        print("Tự động bỏ qua để chuyển sang mô hình tiếp theo trong danh sách space...")
        continue

    for data_cfg in INPUT_FILES_CONFIG:
        input_path = data_cfg["input_path"]
        output_dir = data_cfg["output_dir"]
        split_name = get_split_name(input_path)
        is_profile_split = split_name in PROFILE_SPLITS

        os.makedirs(output_dir, exist_ok=True)
        output_prefix = os.path.join(output_dir, f"{model_cfg['name']}")

        expected_paths = _expected_embedding_paths(output_prefix)
        if RESUME_MODE and _all_exists(expected_paths):
            print(f"[RESUME] Skip {model_cfg['name']} | {split_name}: đã có đủ 3 file embedding")
            continue

        print(f"\n[+] Đang xử lý tập dữ liệu: {os.path.basename(input_path)}")
        print(f"    -> Split nhận diện: {split_name}")
        print(f"    -> Dữ liệu nguồn: {input_path}")
        print(f"    -> Tiền tố lưu trữ: {output_prefix}_[strategy].pt")
        if is_profile_split:
            print("    -> [Profiler] BẬT đo lường cho 4 tập test chính")

        extractor.run_extraction(
            parquet_path=input_path,
            seq_type=model_cfg["seq_type"],
            batch_size=model_cfg["batch_size"],
            output_prefix=output_prefix,
            profiler=profiler if is_profile_split else None
        )

        if is_profile_split:
            profiler.export_to_json(FM_PROFILE_JSON, dataset_name=split_name, source_path=input_path)

        torch.cuda.empty_cache()
        gc.collect()
        
    print(f"\n[+] Đang dọn dẹp và giải phóng VRAM cho cấu hình mô hình: {model_cfg['name']}")
    del model_manager
    del extractor
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#" * 50)
print("[THÀNH CÔNG RỰC RỠ] ĐÃ HOÀN TẤT TRÍCH XUẤT ĐA PHƯƠNG THỨC CHO TẤT CẢ CÁC MÔ HÌNH NỀN TẢNG!")
print("#" * 50)

[*] STRICT PRECHECK: quét toàn bộ input trước khi chạy extraction
    - Quy ước index: 0-based
    - DNA fixed position: 300
    - Protein fixed position: 50
[Precheck:dna] train1_full_seq_final.parquet -> tổng 104385 | lỗi 0
[Precheck:protein] train1_full_seq_final.parquet -> tổng 104385 | lỗi 0
[Precheck:dna] train2_full_seq_final.parquet -> tổng 67990 | lỗi 0
[Precheck:protein] train2_full_seq_final.parquet -> tổng 67990 | lỗi 0
[Precheck:dna] train3_full_seq_final.parquet -> tổng 23208 | lỗi 0
[Precheck:protein] train3_full_seq_final.parquet -> tổng 23208 | lỗi 0
[Precheck:dna] val_full_seq_final.parquet -> tổng 13701 | lỗi 0
[Precheck:protein] val_full_seq_final.parquet -> tổng 13701 | lỗi 0
[Precheck:dna] test_full_seq_after_vep_final.parquet -> tổng 6095 | lỗi 0
[Precheck:protein] test_full_seq_after_vep_final.parquet -> tổng 6095 | lỗi 0
[Precheck:dna] clinvarhq_full_seq_after_vep_final.parquet -> tổng 703 | lỗi 0
[Precheck:protein] clinvarhq_full_seq_after_vep_final.parquet ->

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 1261.37it/s]


[!] Da patch get_head_mask cho remote EsmModel (class + instance).
[+] Nạp mô hình thành công!

  [Profiler] Tải trọng: 485,699,306 params | 926.40 MB
[Token Mapping] model=InstaDeepAI/nucleotide-transformer-500m-human-ref | seq_type=dna | strategy=dna_token_diff_fallback
[Token Strategy] nt_v1_500m: dna_token_diff_fallback

[+] Đang xử lý tập dữ liệu: train1_full_seq_final.parquet
    -> Split nhận diện: train1
    -> Dữ liệu nguồn: D:/variant_data/train1_full_seq_final.parquet
    -> Tiền tố lưu trữ: D:/variant_data/fm_embeddings/train1\nt_v1_500m_[strategy].pt
[*] Đang trích xuất: D:/variant_data/train1_full_seq_final.parquet


Inference: 100%|██████████| 1632/1632 [09:23<00:00,  2.90it/s]


[*] Đang lưu các ma trận đặc trưng...
  -> Đã lưu: D:/variant_data/fm_embeddings/train1\nt_v1_500m_cls.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/train1\nt_v1_500m_center.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/train1\nt_v1_500m_mean.pt (Bao gồm E_ref và E_alt)
[+] Hoàn tất trích xuất!


[+] Đang xử lý tập dữ liệu: train2_full_seq_final.parquet
    -> Split nhận diện: train2
    -> Dữ liệu nguồn: D:/variant_data/train2_full_seq_final.parquet
    -> Tiền tố lưu trữ: D:/variant_data/fm_embeddings/train2\nt_v1_500m_[strategy].pt
[*] Đang trích xuất: D:/variant_data/train2_full_seq_final.parquet


...

  [Profiler] Tốc độ thuần Model: 4.75 ms/sample | Peak VRAM: 1469.00 MB
[*] Đang lưu các ma trận đặc trưng...
  -> Đã lưu: D:/variant_data/fm_embeddings/clinvarhq\esmc_600m_cls.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/clinvarhq\esmc_600m_center.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/clinvarhq\esmc_600m_mean.pt (Bao gồm E_ref và E_alt)
[+] Hoàn tất trích xuất!


[+] Đang xử lý tập dữ liệu: uniprot_full_seq_after_vep_final.parquet
    -> Split nhận diện: uniprot
    -> Dữ liệu nguồn: D:/variant_data/uniprot_full_seq_after_vep_final.parquet
    -> Tiền tố lưu trữ: D:/variant_data/fm_embeddings/uniprot\esmc_600m_[strategy].pt
    -> [Profiler] BẬT đo lường cho 4 tập test chính
[*] Đang trích xuất: D:/variant_data/uniprot_full_seq_after_vep_final.parquet


Inference: 100%|██████████| 42/42 [00:07<00:00,  5.71it/s]


  [Profiler] Tốc độ thuần Model: 4.69 ms/sample | Peak VRAM: 1469.00 MB
[*] Đang lưu các ma trận đặc trưng...
  -> Đã lưu: D:/variant_data/fm_embeddings/uniprot\esmc_600m_cls.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/uniprot\esmc_600m_center.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/uniprot\esmc_600m_mean.pt (Bao gồm E_ref và E_alt)
[+] Hoàn tất trích xuất!


[+] Đang xử lý tập dữ liệu: proteingym_full_seq_after_vep_final.parquet
    -> Split nhận diện: proteingym
    -> Dữ liệu nguồn: D:/variant_data/proteingym_full_seq_after_vep_final.parquet
    -> Tiền tố lưu trữ: D:/variant_data/fm_embeddings/proteingym\esmc_600m_[strategy].pt
    -> [Profiler] BẬT đo lường cho 4 tập test chính
[*] Đang trích xuất: D:/variant_data/proteingym_full_seq_after_vep_final.parquet


Inference: 100%|██████████| 46/46 [00:08<00:00,  5.67it/s]


  [Profiler] Tốc độ thuần Model: 4.70 ms/sample | Peak VRAM: 1469.00 MB
[*] Đang lưu các ma trận đặc trưng...
  -> Đã lưu: D:/variant_data/fm_embeddings/proteingym\esmc_600m_cls.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/proteingym\esmc_600m_center.pt (Bao gồm E_ref và E_alt)
  -> Đã lưu: D:/variant_data/fm_embeddings/proteingym\esmc_600m_mean.pt (Bao gồm E_ref và E_alt)
[+] Hoàn tất trích xuất!


[+] Đang dọn dẹp và giải phóng VRAM cho cấu hình mô hình: esmc_600m

##################################################
[THÀNH CÔNG RỰC RỠ] ĐÃ HOÀN TẤT TRÍCH XUẤT ĐA PHƯƠNG THỨC CHO TẤT CẢ CÁC MÔ HÌNH NỀN TẢNG!
##################################################


# MODULE 3: LATENT GEOMETRIC EXTRACTION

In [4]:
GEOM_PROFILE_JSON = r"D:/variant_data/profiling/geom_profiling.json"

geom_calculator = LatentGeometryCalculator(k_neighbors=32, epsilon=1e-8)

print("=" * 80)
print("[*] KHỞI CHẠY QUÁ TRÌNH TRÍCH XUẤT HÌNH HỌC TIỀM ẨN (LVD & LID)")
print("=" * 80)

for model_name in MODELS:
    for pooling in POOLINGS:
        config_id = f"{model_name}_{pooling}"
        print(f"\n[>>>] ĐANG XỬ LÝ TỔ HỢP: {config_id.upper()} [<<<]")

        index_path = f"{INDEX_DIR}/{config_id}.index"
        geom_profiler = GeometryExtractionProfiler(config_id, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        geom_profiler.reset_peak_memory()

        # Resume: nếu đã có toàn bộ geometry output cho mọi split thì skip trọn config
        expected_geom_paths = [f"{GEOM_DIR}/{split}/{config_id}_geom.parquet" for split in ALL_SPLITS]
        if RESUME_MODE and os.path.exists(index_path) and all(os.path.exists(p) for p in expected_geom_paths):
            print(f"[RESUME] Skip config {config_id}: index + geometry outputs đã đủ")
            continue

        # --- BƯỚC A: HUẤN LUYỆN/LOAD GLOBAL FAISS INDEX TỪ 3 TẬP TRAIN ---
        if RESUME_MODE and os.path.exists(index_path):
            print(f"[RESUME] Dùng lại index có sẵn: {index_path}")
            geom_calculator.load_global_index(index_path=index_path, profiler=geom_profiler)
        else:
            delta_train_parts = []

            for train_split in TRAIN_SPLITS:
                train_pt_path = f"{EMB_DIR}/{train_split}/{config_id}.pt"
                if not os.path.exists(train_pt_path):
                    print(f"[!] Thiếu train embedding: {train_pt_path}. Bỏ qua tổ hợp {config_id}.")
                    delta_train_parts = []
                    break

                train_data = torch.load(train_pt_path, weights_only=False)
                e_ref_train = train_data["E_ref"]
                e_alt_train = train_data["E_alt"]
                delta_train_parts.append((e_alt_train - e_ref_train).cpu().numpy().astype(np.float32))

                del train_data, e_ref_train, e_alt_train
                torch.cuda.empty_cache()
                gc.collect()

            if not delta_train_parts:
                continue

            delta_train = np.concatenate(delta_train_parts, axis=0)
            geom_calculator.build_and_save_global_index(delta_train=delta_train, index_path=index_path, profiler=geom_profiler)
            geom_calculator.load_global_index(index_path=index_path, profiler=geom_profiler)

            del delta_train_parts, delta_train
            torch.cuda.empty_cache()
            gc.collect()

        # --- BƯỚC B: TRÍCH XUẤT HÌNH HỌC CHO TOÀN BỘ SPLITS ---
        for split in ALL_SPLITS:
            pt_path = f"{EMB_DIR}/{split}/{config_id}.pt"
            out_parquet_path = f"{GEOM_DIR}/{split}/{config_id}_geom.parquet"

            if RESUME_MODE and os.path.exists(out_parquet_path):
                print(f"    [RESUME] Skip {split}: đã có {out_parquet_path}")
                continue

            if not os.path.exists(pt_path):
                print(f"    [!] Bỏ qua tập {split} vì không thấy file {pt_path}")
                continue

            print(f"  -> Đang trích xuất đặc trưng cho tập: {split.upper()}...")
            is_profile_split = split in PROFILE_SPLITS
            if is_profile_split:
                geom_profiler._acc["lvd_time_s"] = 0.0
                geom_profiler._acc["lid_time_s"] = 0.0
                geom_profiler._acc["geometry_total_time_s"] = 0.0
                geom_profiler.reset_peak_memory()
            split_data = torch.load(pt_path, weights_only=False)
            metadata = split_data["metadata"]
            llr = split_data["llr"].cpu().numpy().flatten()
            e_ref = split_data["E_ref"].cuda()
            e_alt = split_data["E_alt"].cuda()

            geom_features = geom_calculator.extract_geometry_features(
                e_ref,
                e_alt,
                profiler=geom_profiler if is_profile_split else None
            )

            df_geom = pd.DataFrame({
                "Variant_ID": metadata,
                "LLR": llr.astype(np.float32),
                "LVD_L2": geom_features["LVD_L2"].cpu().numpy().flatten().astype(np.float32),
                "LVD_Cosine": geom_features["LVD_Cosine"].cpu().numpy().flatten().astype(np.float32),
                "LID": geom_features["LID"].cpu().numpy().flatten().astype(np.float32)
            })

            df_geom.to_parquet(out_parquet_path, index=False)
            print(f"      [+] Đã lưu: {out_parquet_path} (Kích thước: {df_geom.shape})")

            if is_profile_split:
                geom_profiler.finalize(num_samples=len(metadata))
                geom_profiler.export_to_json(
                    GEOM_PROFILE_JSON,
                    split_name=split,
                    source_path=pt_path
                )

            del split_data, e_ref, e_alt, geom_features, df_geom
            torch.cuda.empty_cache()
            gc.collect()

        geom_calculator.index = None
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "=" * 80)
print("[THÀNH CÔNG] ĐÃ HOÀN TẤT TOÀN BỘ QUÁ TRÌNH TRÍCH XUẤT HÌNH HỌC VÀ LƯU THÀNH PARQUET!")
print("=" * 80)

[FAISS] GPU APIs khong kha dung, fallback sang CPU mode
[*] KHỞI CHẠY QUÁ TRÌNH TRÍCH XUẤT HÌNH HỌC TIỀM ẨN (LVD & LID)

[>>>] ĐANG XỬ LÝ TỔ HỢP: NT_V1_500M_CLS [<<<]
[*] Đang khởi tạo FAISS Index không gian 1280 chiều...
[*] Đang nạp 195583 vector vào CPU Index...
[+] Đã huấn luyện và lưu Global FAISS Index tại: D:/variant_data/faiss_indexes/nt_v1_500m_cls.index

[+] Đã nạp thành công CPU Index từ: D:/variant_data/faiss_indexes/nt_v1_500m_cls.index
  -> Đang trích xuất đặc trưng cho tập: TRAIN1...
      [+] Đã lưu: D:/variant_data/geometry/train1/nt_v1_500m_cls_geom.parquet (Kích thước: (104385, 5))
  -> Đang trích xuất đặc trưng cho tập: TRAIN2...
      [+] Đã lưu: D:/variant_data/geometry/train2/nt_v1_500m_cls_geom.parquet (Kích thước: (67990, 5))
  -> Đang trích xuất đặc trưng cho tập: TRAIN3...
      [+] Đã lưu: D:/variant_data/geometry/train3/nt_v1_500m_cls_geom.parquet (Kích thước: (23208, 5))
  -> Đang trích xuất đặc trưng cho tập: VAL...
      [+] Đã lưu: D:/variant_data/geome

# MODULE 4: BIOLOGICAL CONTEXT NORMALIZATION

In [5]:
normalizer = AdvancedFeatureNormalizer(epsilon=1e-8)

print("=" * 80)
print("[*] KHỞI CHẠY QUÁ TRÌNH CHUẨN HÓA DỮ LIỆU BẢNG (BIO & GEOMETRY)")
print("=" * 80)

# ==============================================================================
# PHẦN 1: CHUẨN HÓA DỮ LIỆU SINH HỌC (BIO CONTEXT)
# ==============================================================================
train_merged_bio_out = f"{OUTPUT_DIR}/train_merged_normalized.parquet"

if RESUME_MODE and os.path.exists(train_merged_bio_out):
    print(f"[RESUME] Skip fit Bio: đã có {train_merged_bio_out}")
else:
    print("  -> Nạp và gộp 3 tập Train (Bio) để fit normalizer...")
    bio_train_dfs = []
    for p in BIO_TRAIN_PATHS:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Thiếu file train bio: {p}")
        bio_train_dfs.append(pd.read_parquet(p))

    df_train_bio = pd.concat(bio_train_dfs, axis=0, ignore_index=True)
    df_train_bio_norm = normalizer.fit_transform_bio(df_train_bio, ARTIFACTS_DIR)
    df_train_bio_norm.to_parquet(train_merged_bio_out, index=False)

for p in BIO_TRAIN_PATHS:
    split_name = os.path.basename(p).replace("_full_seq_final.parquet", "")
    out_path = f"{OUTPUT_DIR}/{split_name}_normalized.parquet"
    if RESUME_MODE and os.path.exists(out_path):
        print(f"  [RESUME] Skip train bio split {split_name}: đã có file")
        continue
    df_split = pd.read_parquet(p)
    df_split_norm = normalizer.transform_bio(df_split, ARTIFACTS_DIR)
    df_split_norm.to_parquet(out_path, index=False)

print("  -> Transform Bio cho Val + 4 tập Test chính...")
for eval_path in BIO_EVAL_PATHS:
    if not os.path.exists(eval_path):
        print(f"    [!] Bỏ qua vì thiếu file: {eval_path}")
        continue
    out_path = f"{OUTPUT_DIR}/{os.path.basename(eval_path).replace('.parquet', '_normalized.parquet')}"
    if RESUME_MODE and os.path.exists(out_path):
        print(f"    [RESUME] Skip {os.path.basename(eval_path)}: đã có file")
        continue
    df_eval_bio = pd.read_parquet(eval_path)
    df_eval_bio_norm = normalizer.transform_bio(df_eval_bio, ARTIFACTS_DIR)
    df_eval_bio_norm.to_parquet(out_path, index=False)

# ==============================================================================
# PHẦN 2: CHUẨN HÓA DỮ LIỆU HÌNH HỌC (GEOMETRIC CONTEXT)
# ==============================================================================
print("\n[>>>] PHẦN 2: CHUẨN HÓA ĐẶC TRƯNG HÌNH HỌC [<<<]")

for model_name in MODELS:
    for pooling in POOLINGS:
        config_name = f"{model_name}_{pooling}"
        print(f"\n  [Tổ hợp: {config_name.upper()}]")

        out_train_geom_merged = f"{GEOM_DIR}/train_merged/{config_name}_geom_norm.parquet"
        os.makedirs(f"{GEOM_DIR}/train_merged", exist_ok=True)

        need_fit = not (RESUME_MODE and os.path.exists(out_train_geom_merged))
        if need_fit:
            train_geom_dfs = []
            for train_split in TRAIN_SPLITS:
                train_geom_path = f"{GEOM_DIR}/{train_split}/{config_name}_geom.parquet"
                if not os.path.exists(train_geom_path):
                    print(f"    [!] Thiếu train geometry: {train_geom_path}. Bỏ qua tổ hợp này.")
                    train_geom_dfs = []
                    break
                train_geom_dfs.append(pd.read_parquet(train_geom_path))

            if not train_geom_dfs:
                continue

            print("    -> Fit & Transform trên train1+train2+train3...")
            df_train_geom_merged = pd.concat(train_geom_dfs, axis=0, ignore_index=True)
            df_train_geom_merged_norm = normalizer.fit_transform_geom(df_train_geom_merged, ARTIFACTS_DIR, config_name)
            df_train_geom_merged_norm.to_parquet(out_train_geom_merged, index=False)
            del df_train_geom_merged, df_train_geom_merged_norm, train_geom_dfs
            gc.collect()
        else:
            print("    [RESUME] Skip fit geometry: đã có train_merged normalized")

        # Transform lại từng train split theo scaler đã fit
        for train_split in TRAIN_SPLITS:
            train_geom_path = f"{GEOM_DIR}/{train_split}/{config_name}_geom.parquet"
            out_split = f"{GEOM_DIR}/{train_split}/{config_name}_geom_norm.parquet"
            if RESUME_MODE and os.path.exists(out_split):
                print(f"    [RESUME] Skip train split {train_split}: đã có geom_norm")
                continue
            if not os.path.exists(train_geom_path):
                print(f"    [!] Không thấy {train_geom_path}, bỏ qua.")
                continue
            df_train_split = pd.read_parquet(train_geom_path)
            df_train_split_norm = normalizer.transform_geom(df_train_split, ARTIFACTS_DIR, config_name)
            df_train_split_norm.to_parquet(out_split, index=False)

        for split in VAL_SPLITS + TEST_SPLITS:
            eval_geom_path = f"{GEOM_DIR}/{split}/{config_name}_geom.parquet"
            out_eval_geom = f"{GEOM_DIR}/{split}/{config_name}_geom_norm.parquet"
            if RESUME_MODE and os.path.exists(out_eval_geom):
                print(f"    [RESUME] Skip {split}: đã có geom_norm")
                continue
            if not os.path.exists(eval_geom_path):
                print(f"    [!] Không thấy {eval_geom_path}, bỏ qua.")
                continue

            print(f"    -> Transform {split.upper()}...")
            df_eval_geom = pd.read_parquet(eval_geom_path)
            df_eval_geom_norm = normalizer.transform_geom(df_eval_geom, ARTIFACTS_DIR, config_name)
            df_eval_geom_norm.to_parquet(out_eval_geom, index=False)
            del df_eval_geom, df_eval_geom_norm
            gc.collect()

print("\n" + "=" * 80)
print("[THÀNH CÔNG] DỮ LIỆU ĐÃ ĐƯỢC CHUẨN HÓA TOÀN DIỆN VÀ SẴN SÀNG CHO FUSION (MODULE 5)!")
print("=" * 80)

[*] KHỞI CHẠY QUÁ TRÌNH CHUẨN HÓA DỮ LIỆU BẢNG (BIO & GEOMETRY)
  -> Nạp và gộp 3 tập Train (Bio) để fit normalizer...
[*] Đang học phân phối và chuẩn hóa tập Train (195583 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (104385 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (67990 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (23208 variants)...
  -> Transform Bio cho Val + 4 tập Test chính...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (13701 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (6095 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (703 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (1333 variants)...
[*] Đang nạp Artifacts để chuẩn hóa tập Val/Test (1472 variants)...

[>>>] PHẦN 2: CHUẨN HÓA ĐẶC TRƯNG HÌNH HỌC [<<<]

  [Tổ hợp: NT_V1_500M_CLS]
    -> Fit & Transform trên train1+train2+train3...
    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> T

c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_

    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: NT_V3_650M_CENTER]
    -> Fit & Transform trên train1+train2+train3...


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]


    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: NT_V3_650M_MEAN]
    -> Fit & Transform trên train1+train2+train3...


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_

    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log10
  new_mgr_locs = self._mgr_locs[indices]


    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: NT_V2_500M_CLS]
    -> Fit & Transform trên train1+train2+train3...
    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: NT_V2_500M_CENTER]
    -> Fit & Transform trên train1+train2+train3...
    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: NT_V2_500M_MEAN]
    -> Fit & Transform trên train1+train2+train3...
    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: ESM1B_650M_CLS]
    -> Fit & Transform trên train1+train2+train3...
    -> Transform VAL...
    -> Transform TEST...
    -> Transform CLINVARHQ...
    -> Transform UNIPROT...
    -> Transform PROTEINGYM...

  [Tổ hợp: ESM1B_650M_CENTER]
    -> Fit & Transform trên train1